# Compare wake-word models: personal and general

Scores every `server/models/jarvie_<MODE>*.onnx` on held-out real recordings, per speaker. Set `MODE` below to
`wake` (stage 1), `name` (stage 2, "jarvie" alone) or `sleep`:
- **personal**: your test set (`~/jarvie-data/audio/<MODE>/`, recorded with `record_session.py --split test`)
- **general**: other people's test sets (`~/jarvie-data/audio-speakers/<name>/<MODE>/`, `--split test --speaker <name>`),
  averaged so every speaker counts equally. Use people who are in **no** training data.

Same definitions as `wakeword_eval.ipynb`: each model's threshold is the equal-error-rate threshold on normal-pace
clips; fast/slow recall and how often each similarity tier fires are measured at that threshold. Clips are padded with
1 s of silence on both sides, as livekit's `predict_clip` does. The run refuses to score a test clip that also appears in
`~/jarvie-data/real/` (training data).

Model families (from `train_wakeword_runpod.ipynb`): `B` synthetic (M1), `B+real-v1` synthetic + your recordings (M2),
`B+real-v1-nosyn` recordings only (M3), `B+real-v1-syn10__ft` B fine-tuned on recordings (M4).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

SERVER = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "models").is_dir() and (p / "jarvie").is_dir()
)
sys.path.insert(0, str(SERVER))
from jarvie.eval import compare  # noqa: E402

pd.set_option("display.precision", 3)
# Which model family: "wake" (stage 1: "jarvie, wake up"), "name" (stage 2: "jarvie" alone) or "sleep".
MODE = "wake"
MODELS = sorted((SERVER / "models").glob(f"jarvie_{MODE}*.onnx"))
TEST_SETS = compare.discover_test_sets(MODE)  # {"me": ..., "<other speaker>": ...}
TRAIN_ROOT = Path.home() / "jarvie-data" / "real"  # checked for overlap with the test sets

# Clips that say "jarvie, I need some help": the old model was trained on it, A/B weren't (they use
# "jarvie, help me"), so including them compares the models on a phrase only one of them knows.
EXCLUDE = (
    {f"pos_desk_loud-normal_wake_{i:03d}.wav" for i in range(4, 8)} if MODE == "wake" else set()
)
# Negatives that are really acceptable wake phrases, scored as positives: put the "jarvie wakie wake"
# clips here, and the "jarvis wake up" ones too if Jarvis should wake Jarvie.
AS_POSITIVE: set[str] = set()

print("models:", *[m.name for m in MODELS], sep="\n  ")
print("test sets:", *[f"{k}: {v}" for k, v in TEST_SETS.items()], sep="\n  ")

In [ ]:
compare.check_leakage(TEST_SETS, TRAIN_ROOT)
scores = compare.score_test_sets(MODELS, TEST_SETS)
scores = compare.apply_labels(scores, EXCLUDE, AS_POSITIVE)
scores.groupby("speaker").apply(compare.counts).unstack(0).fillna(0).astype(int)

## Personal: your voice
Lower `eer_normal` and `fires_*` are better; higher recall is better. This table picks the **personal model**.

In [ ]:
personal = compare.summarize(scores[scores.speaker == compare.OWNER])
personal.T

## General: other people's voices
Averaged over speakers. This table picks the **deployment model**; it should beat the synthetic-only model here.
Empty until you have test recordings from at least one other person.

In [ ]:
general = compare.general_summary(scores)
general.T if not general.empty else "no other speakers' test recordings yet"

In [ ]:
# Every speaker separately: spot a model that works for some voices and not others.
compare.summarize_by_speaker(scores)[
    [
        "threshold",
        "eer_normal",
        "recall_normal",
        "recall_fast",
        "recall_slow",
        "fires_similar",
        "fires_very-similar",
    ]
]

## Per-clip scores\nWhere models disagree, listen to the clip: it usually explains the difference.

In [ ]:
per_clip = scores.pivot_table(
    index=["speaker", "kind", "similarity", "pace", "file"], columns="model", values="max_score"
)
per_clip.style.background_gradient(axis=None, cmap="RdYlGn")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5), squeeze=False)
cols = ["recall_normal", "recall_fast", "recall_slow", "fires_similar", "fires_very-similar"]
for ax, (title, table) in zip(
    axes[0], [("Personal (your voice)", personal), ("General (other voices)", general)]
):
    if table.empty:
        ax.set_axis_off()
        ax.set_title(f"{title}: no data yet")
        continue
    table[cols].T.plot.bar(ax=ax, rot=0, legend=False)
    ax.set(title=title, ylim=(0, 1))
axes[0][0].legend(fontsize=7)
plt.tight_layout()
plt.show()